
# 11 机器学习与向前滚动验证：模型为什么会“记住”历史

> **本章要解决的问题**：一个模型在训练集上 90% 准确，样本外却亏钱，为什么？树、SVM、神经网络、注意力、强化学习，在金融数据上最容易犯什么错？怎样做“不偷看未来”的验证？

## 本章导读

机器学习不是“换个更高级的模型”，而是**用数据拟合一个从输入到输出的规则**。在金融里，这个规则有三个天然敌人：

1. **信噪比低**：价格里可预测的部分很小；
2. **非平稳**：今天的关系明天可能变；
3. **偷看未来**：标签、特征、验证方式，任何一处泄漏都会让回测虚高。

本章分九节：

1. 监督学习、过拟合、偏差-方差；
2. 非线性拟合：直线不够时怎么办；
3. 神经网络与激活函数；
4. 决策树与随机森林；
5. 支持向量机；
6. 聚类；
7. 注意力机制；
8. 强化学习；
9. 金融 ML 的特殊性与向前滚动验证。

### 学习目标

- 能切训练/验证/测试，并识别泄漏。
- 能拟合线性、树、MLP，并读懂过拟合曲线。
- 能用 purged/embargo 做时序验证。
- 能解释特征重要性/注意力，但不把它当因果。
- 能把预测转成因子或仓位，并回测。

### 先修

第02章的概率/统计；第04章的矩阵乘法；第07章的最优化/梯度下降；会一点 Python 和 sklearn。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="11 机器学习与向前滚动验证" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>11 机器学习与向前滚动验证｜知识图</title>
<desc>本章主线：时间不能打乱——训练、验证、测试按时间切，purge/embargo 挡住泄漏。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a10" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao10" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag10" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">11</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">机器学习与向前滚动验证</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：时间不能打乱——训练、验证、测试按时间切，purge/embargo 挡住泄漏。</text>
<rect x="45" y="95" width="160" height="53" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="125" y="114" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">训练集</text>
<text x="125" y="133" font-size="9.5" fill="#6E7B8B" text-anchor="middle">拟合参数</text>
<rect x="45" y="158" width="160" height="53" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="125" y="177" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">purge / embargo</text>
<text x="125" y="196" font-size="9.5" fill="#6E7B8B" text-anchor="middle">挖掉重叠标签</text>
<rect x="45" y="221" width="160" height="53" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="125" y="240" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">验证集</text>
<text x="125" y="259" font-size="9.5" fill="#6E7B8B" text-anchor="middle">选模型、调超参</text>
<rect x="45" y="284" width="160" height="53" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="125" y="303" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">测试集</text>
<text x="125" y="322" font-size="9.5" fill="#6E7B8B" text-anchor="middle">只看一次，不回头</text>
<path d="M 125 148 L 125 158" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a10)"/>
<path d="M 125 211 L 125 221" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a10)"/>
<path d="M 125 274 L 125 284" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a10)"/>
<text x="235" y="110" font-size="11" fill="#1B3A5C" font-weight="bold">向前滚动切分：不打乱、不回头</text>
<text x="242" y="154" font-size="9" fill="#6E7B8B" text-anchor="end">Fold 1</text>
<rect x="250" y="140" width="120" height="26" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<rect x="370" y="140" width="12" height="26" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="2 2"/>
<rect x="382" y="140" width="52" height="26" fill="#E07B39" rx="4"/>
<rect x="434" y="140" width="311" height="26" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4" stroke-dasharray="5 4"/>
<text x="260" y="157" font-size="8.8" fill="#1B3A5C">训练</text>
<text x="442" y="157" font-size="8.2" fill="#6E7B8B">未来不可见</text>
<text x="242" y="200" font-size="9" fill="#6E7B8B" text-anchor="end">Fold 2</text>
<rect x="250" y="186" width="186" height="26" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<rect x="436" y="186" width="12" height="26" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="2 2"/>
<rect x="448" y="186" width="52" height="26" fill="#E07B39" rx="4"/>
<rect x="500" y="186" width="245" height="26" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4" stroke-dasharray="5 4"/>
<text x="260" y="203" font-size="8.8" fill="#1B3A5C">训练</text>
<text x="508" y="203" font-size="8.2" fill="#6E7B8B">未来不可见</text>
<text x="242" y="246" font-size="9" fill="#6E7B8B" text-anchor="end">Fold 3</text>
<rect x="250" y="232" width="252" height="26" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<rect x="502" y="232" width="12" height="26" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="2 2"/>
<rect x="514" y="232" width="52" height="26" fill="#E07B39" rx="4"/>
<rect x="566" y="232" width="179" height="26" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4" stroke-dasharray="5 4"/>
<text x="260" y="249" font-size="8.8" fill="#1B3A5C">训练</text>
<text x="574" y="249" font-size="8.2" fill="#6E7B8B">未来不可见</text>
<text x="242" y="292" font-size="9" fill="#6E7B8B" text-anchor="end">Fold 4</text>
<rect x="250" y="278" width="318" height="26" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<rect x="568" y="278" width="12" height="26" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="2 2"/>
<rect x="580" y="278" width="52" height="26" fill="#E07B39" rx="4"/>
<rect x="632" y="278" width="113" height="26" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4" stroke-dasharray="5 4"/>
<text x="260" y="295" font-size="8.8" fill="#1B3A5C">训练</text>
<text x="640" y="295" font-size="8.2" fill="#6E7B8B">未来不可见</text>
<path d="M 250 336 L 748 336" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag10)"/>
<line x1="250" y1="332" x2="250" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="250" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t0</text>
<line x1="349" y1="332" x2="349" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="349" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t1</text>
<line x1="448" y1="332" x2="448" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="448" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t2</text>
<line x1="547" y1="332" x2="547" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="547" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t3</text>
<line x1="646" y1="332" x2="646" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="646" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t4</text>
<line x1="745" y1="332" x2="745" y2="340" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="745" y="353" font-size="8.5" fill="#6E7B8B" text-anchor="middle">t5</text>
<rect x="250" y="366" width="10" height="10" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1" rx="2"/>
<text x="265" y="375" font-size="8.8" fill="#6E7B8B">训练</text>
<rect x="328" y="366" width="10" height="10" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="2 2"/>
<text x="343" y="375" font-size="8.8" fill="#6E7B8B">purge</text>
<rect x="406" y="366" width="10" height="10" fill="#E07B39" stroke="#E07B39" stroke-width="1" rx="2"/>
<text x="421" y="375" font-size="8.8" fill="#6E7B8B">验证</text>
<rect x="484" y="366" width="10" height="10" fill="#FFFFFF" stroke="#6E7B8B" stroke-width="1" rx="2" stroke-dasharray="4 3"/>
<text x="499" y="375" font-size="8.8" fill="#6E7B8B">未来不可见</text>
<text x="600" y="375" font-size="9.2" fill="#1B3A5C">purge / embargo：挡住重叠标签与泄漏</text>
</svg>

> **图**：本章主线：时间不能打乱——训练、验证、测试按时间切，purge/embargo 挡住泄漏。


In [1]:

import os
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("环境准备完成")


环境准备完成



## 在量化研究中的位置

- **谁用**：研究员、因子组合、预测建模。
- **输入**：特征、标签、时间索引。
- **输出**：预测、验证曲线、特征重要性、可解释性。
- **在流程的哪一步**：非线性建模与因子组合。
- **失败模式**：随机切分、标签泄漏、过拟合、把重要性当因果。
- **最小工作流**：时间切分 → 训练/验证 → purged/embargo → 解释 → 回测。

---
## 1. 监督学习、过拟合、偏差-方差

### 1.1 四个词

- **特征** $X$：模型能看到的输入（动量、成交量、波动率）；
- **标签** $y$：要预测的结果（未来收益、涨跌、是否延续）；
- **训练集**：用来拟合参数；
- **验证/测试集**：用来选模型、评估样本外。

模型在训练集上好，不代表在测试集上好。**过拟合**就是“记住了训练集的噪声，没学到可迁移的规律”。

### 1.2 偏差-方差

预测误差可以拆成：

$$\text{误差}=\text{偏差}^2+\text{方差}+\text{不可约噪声}.$$

**公式拆解**

- **符号**：误差=新样本期望平方误差；偏差=$E[\hat f]-f$；方差=模型对训练集扰动的敏感度。
- **结构**：总误差=偏差²+方差+噪声；噪声不可约。
- **代入**：A 训练 0.01/测试 0.10，B 训练 0.05/测试 0.06；A 测试更差，方差更大（过拟合）。
- **边界**：模型再复杂也降不到 0；偏差与方差通常此消彼长。
- **用途**：选模型/正则；训练误差低不等于样本外好。

- 模型太简单：偏差大（欠拟合）；
- 模型太复杂：方差大（过拟合）；
- 数据噪声：不可约。

金融数据噪声很大，所以“稍微简单一点、正则强一点”的模型，往往样本外更好。

### 1.3 代码：多项式次数与过拟合



In [2]:

import numpy as np

rng = np.random.default_rng(1001)
x = np.linspace(-1, 1, 30)
y = x ** 2 + rng.normal(0, 0.1, size=30)

x_train, y_train = x[:20], y[:20]
x_test, y_test = x[20:], y[20:]

for degree in [1, 2, 4]:
    coef = np.polyfit(x_train, y_train, degree)
    train_mse = np.mean((np.polyval(coef, x_train) - y_train) ** 2)
    test_mse = np.mean((np.polyval(coef, x_test) - y_test) ** 2)
    print("次数", degree, "训练 MSE", round(train_mse, 4), "测试 MSE", round(test_mse, 4))


次数 1 训练 MSE 0.025 测试 MSE 0.9512
次数 2 训练 MSE 0.008 测试 MSE 0.0082
次数 4 训练 MSE 0.0056 测试 MSE 1.4057



**读数与应用**：1 次欠拟合，2 次接近真实关系，4 次训练误差更小但测试误差变大——这就是过拟合。真实数据里，次数越高、特征越多，越要正则化和验证。

**常见错误**：

1. 用测试集调参。测试集只能用一次。
2. 只看训练误差。训练误差低不代表样本外好。
3. 用随机打乱的 K 折处理时间序列。未来信息会泄漏到训练。

**停下来想一想**：为什么金融数据里“过拟合”比图像识别更严重？

<details><summary>对照你的思路</summary>

金融信噪比低：可预测部分很小，噪声很大；样本量有限、非平稳；研究人员会反复试特征/参数，产生多重检验。模型很容易记住噪声，样本外失效。图像/文本的样本量大、关系相对稳定，过拟合的代价小一些。

</details>

---



## 2. 非线性拟合：直线不够时怎么办

### 2.1 为什么直线不够

若 $y$ 与 $x$ 的关系是 U 形（比如“波动太大/太小都不好”），一条直线无法同时拟合两端。解决办法：

- 加多项式特征 $x^2,x^3$；
- 加交互项 $x_1x_2$；
- 用树/核方法/神经网络。

### 2.2 线性模型也能拟合曲线

只要把特征变换成非线性的，模型对原始 $x$ 就是非线性的，但待估系数仍是线性的：

$$y=\beta_0+\beta_1x+\beta_2x^2.$$

这就是“基函数展开”。它仍然是线性回归，可以用最小二乘解。

### 2.3 过拟合的控制

- 限制多项式次数；
- 加 L1/L2 正则；
- 用验证集选复杂度；
- 增加样本/减少特征。

**常见错误**：

1. 一味加高次项，训练误差下降、测试误差上升。
2. 不标准化就加正则；惩罚对尺度敏感。
3. 用全样本选次数，再回测同一段。

**停下来想一想**：为什么“加特征”不总是提升模型？

<details><summary>对照你的思路</summary>

加特征会增加方差。如果新特征和标签的关系很弱，模型会拟合噪声，样本外变差。金融里特征很多但信噪比低，特征选择和正则比“堆特征”更重要。

</details>

---



## 3. 神经网络与激活函数

### 3.1 一个神经元

$$z=\sum_i w_ix_i+b,\qquad a=\sigma(z).$$

- $w$：权重；$b$：偏置；$\sigma$：激活函数；
- 没有激活函数，多层线性变换仍等价于一层线性变换。

### 3.2 常见激活

| 激活 | 公式 | 特点 |
|---|---|---|
| ReLU | $\max(0,z)$ | 简单、稀疏、可能“死神经元” |
| tanh | $(e^z-e^{-z})/(e^z+e^{-z})$ | 输出在 $[-1,1]$，中心化 |
| sigmoid | $1/(1+e^{-z})$ | 输出在 $[0,1]$，适合概率 |

### 3.3 反向传播

前向算出预测，损失量化误差，反向沿计算图求每个参数的梯度，再用梯度下降更新。初学可以只跟踪一个权重，避免被层数分散注意力。

### 3.4 XOR

XOR 不是线性可分的：一条直线分不开 $(0,0),(1,1)$ 和 $(0,1),(1,0)$。加一个隐藏层 + 非线性激活，就能分开。

### 3.5 代码：小神经网络



In [3]:

import numpy as np
from sklearn.neural_network import MLPClassifier

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

clf = MLPClassifier(hidden_layer_sizes=(4,), activation="relu",
                    max_iter=5000, random_state=10).fit(X, y)
print("预测", clf.predict(X))
print("概率", clf.predict_proba(X).round(2))


预测 [0 1 1 0]
概率 [[0.87 0.13]
 [0.24 0.76]
 [0.05 0.95]
 [0.94 0.06]]



**读数与应用**：4 个隐藏单元、ReLU，就能学会 XOR。真实金融网络更大、更深，但核心还是“线性组合 + 非线性激活 + 反向传播”。

**常见错误**：

1. 不标准化输入。不同尺度会让训练不稳。
2. 网络太大、样本太少，过拟合。
3. 用未来数据做特征/标准化；金融里比图像更致命。

**停下来想一想**：为什么没有激活函数的多层网络等价于一层？

<details><summary>对照你的思路</summary>

两层线性变换：$W_2(W_1x+b_1)+b_2=(W_2W_1)x+(W_2b_1+b_2)$，仍是线性变换。加非线性激活后，复合函数才可能弯曲。深度带来的是“分段线性/非线性”的表达能力。

</details>

---



## 4. 决策树与随机森林

### 4.1 树在做什么

每次选一个特征、一个阈值，把样本切成两堆，让每堆的标签更纯。常用不纯度：

$$\text{Gini}=1-\sum_k p_k^2.$$

切分后按各堆样本数加权，选让不纯度下降最多的切分。

### 4.2 深度与过拟合

树越深，越容易记住训练集；限制 `max_depth`、`min_samples_leaf` 可以正则化。随机森林用很多棵树 + 随机特征，降低方差。

### 4.3 特征重要性

树可以给“每个特征让不纯度下降多少”的分数。但重要性高不等于因果，也不等于样本外有用；相关特征会分摊重要性。

### 4.4 代码：一棵浅树



In [4]:

import numpy as np
from sklearn.tree import DecisionTreeClassifier

X = np.array([[1.0], [2.0], [8.0], [9.0]])
y = np.array([0, 0, 1, 1])

tree = DecisionTreeClassifier(max_depth=1, random_state=10).fit(X, y)
print("切分阈值", tree.tree_.threshold[0])
print("预测 3、7", tree.predict([[3.0], [7.0]]))


切分阈值 5.0
预测 3、7 [0 1]



**读数与应用**：阈值 5 把两类分开；`max_depth=1` 就是“一个 stump”。真实数据里，单棵树容易过拟合，常用随机森林/梯度提升。

**常见错误**：

1. 树太深，记住噪声。
2. 用特征重要性代替因果/稳定性检验。
3. 不做时间序列切分，随机抽样导致泄漏。

**停下来想一想**：为什么随机森林比单棵树稳？

<details><summary>对照你的思路</summary>

单棵树方差大：换一批样本，树结构可能大变。随机森林对样本/特征做随机化，训练很多棵树再平均，方差下降；代价是解释性变差、计算变贵。

</details>

---



## 5. 支持向量机

### 5.1 最大间隔

同样能分开两类的线有很多条；SVM 选“离最近样本最远”的那条。最近样本叫支持向量。

### 5.2 软间隔

真实数据有重叠，允许少量误分：

$$\min \frac12||w||^2+C\sum_i\xi_i.$$

**公式拆解**

- **符号**：$w$ 法向量；$\xi_i\ge0$ 越界程度；$C$ 越大越不容忍误分。
- **结构**：第一项最大化间隔，第二项惩罚越界。
- **代入**：$C=0.1$ 间隔宽、偏差大；$C=100$ 间隔窄、过拟合风险高。
- **边界**：$C\to0$ 欠拟合；$C\to\infty$ 硬间隔、噪声敏感；特征要标准化。
- **用途**：分类/核方法；$C$ 是正则强度的反向旋钮。

$C$ 越大，越不容忍误分。
- **代入**：$C=0.1$ 时，模型宁可少数样本越界也要宽间隔，偏差上升；$C=100$ 时，几乎不允许越界，间隔变窄，方差/过拟合风险上升。
- **边界**：$C\to0$ 退化为“只求最大间隔”，容易欠拟合；$C\to\infty$ 退化为硬间隔，噪声敏感；特征未标准化时，$||w||$ 被大尺度特征主导。
- **用途/误读**：用于分类/非线性核方法；$C$ 是正则强度的反向旋钮。不能把“训练集零误分”当成好模型。

$C$ 越大，越不容忍误分，越容易过拟合；$C$ 越小，间隔越宽，偏差越大。

### 5.3 核函数

线性分不开时，用核函数把样本映射到高维，在高维找线性分界。常用 RBF 核。

### 5.4 代码：线性 SVM



In [5]:

import numpy as np
from sklearn.svm import SVC

X = np.array([[-2.0], [-1.0], [1.0], [3.0]])
y = np.array([-1, -1, 1, 1])

for C in [0.1, 100]:
    model = SVC(kernel="linear", C=C).fit(X, y)
    boundary = -model.intercept_[0] / model.coef_[0, 0]
    print("C", C, "支持向量", model.support_vectors_.ravel(), "分界", round(boundary, 3))


C 0.1 支持向量 [-2. -1.  1.  3.] 分界 0.5
C 100 支持向量 [-1.  1.] 分界 0.0



**读数与应用**：$C$ 很小时，间隔宽、允许误分；$C$ 很大时，尽量把所有点分对，边界可能被异常值拉偏。SVM 对尺度敏感，先标准化。

**常见错误**：

1. 不标准化。距离会被大尺度特征主导。
2. 把 $C$ 当“正则强度”。$C$ 越大，正则越弱。
3. 用核 SVM 处理几万个特征/样本，计算爆炸。

**停下来想一想**：为什么 SVM 的解只由支持向量决定？

<details><summary>对照你的思路</summary>

最大间隔的约束只被“最近的点”绑定；其他点离边界更远，移动它们不影响最优边界。所以解只依赖支持向量。这也是 SVM 对异常值敏感的原因：一个远离的异常点可能变成支持向量，拉动边界。

</details>

---



## 6. 聚类：没有标签时怎样分组

### 6.1 KMeans

把样本分成 $K$ 组，最小化组内平方距离：

$$\min_{\{C_k\}}\sum_{k=1}^K\sum_{x\in C_k}||x-\mu_k||^2.$$

交替两步：

1. 把每个样本分给最近的中心；
2. 把中心改成组内均值。

### 6.2 标准化

距离对尺度敏感。成交量（手）和收益率（小数）放一起，聚类会被成交量主导。先 z-score，再聚类。

### 6.3 选择 K

- 肘部法：看 inertia 随 K 的下降；
- 轮廓系数；
- 业务可解释性。

没有“唯一正确”的 K；要看聚类是否稳定、是否有用。

### 6.4 代码：KMeans



In [6]:

import numpy as np
from sklearn.cluster import KMeans

X = np.array([[1.0], [2.0], [8.0], [9.0]])
km = KMeans(n_clusters=2, n_init=10, random_state=10).fit(X)
print("中心", np.sort(km.cluster_centers_.ravel()))
print("组内平方和", round(km.inertia_, 3))


中心 [1.5 8.5]
组内平方和 1.0



**读数与应用**：两个中心大约 1.5 和 8.5，组内平方和 1.0。真实资产聚类里，中心不是“价格”，而是标准化后的特征均值。

**常见错误**：

1. 不标准化。大尺度特征主导距离。
2. 用价格水平聚类。价格高的资产会聚在一起，和相似性无关。
3. 只跑一次 KMeans。初始中心不同，结果可能不同；多起点/稳定性检查。

**停下来想一想**：为什么价格水平不适合做聚类特征？

<details><summary>对照你的思路</summary>

价格水平受品种、合约乘数、通胀影响；两个价格接近的资产可能完全不相关。聚类应该用收益、波动率、相关性等“可比”的特征，并标准化。价格水平聚类往往只反映“名义价格”。

</details>

---



## 7. 注意力机制：按问题分配权重

### 7.1 Q、K、V

- 查询 $Q$：你要找什么；
- 键 $K$：每条信息用什么匹配；
- 值 $V$：匹配后取出的内容。

打分：

$$\text{score}_i=\frac{Q\cdot K_i}{\sqrt{d}}.$$

除以 $\sqrt d$ 是防止维度大时点积过大、softmax 饱和。

### 7.2 Softmax

$$\text{score}_i=\frac{Q\cdot K_i}{\sqrt{d}},\qquad w_i=\frac{e^{s_i}}{\sum_j e^{s_j}}.$$

**公式拆解**

- **符号**：$Q$ 是当前查询向量；$K_i$ 是第 $i$ 个键；$d$ 是向量维度；$s_i$ 是打分；$w_i$ 是归一化权重。
- **结构**：先做点积 $Q\cdot K_i$ 衡量“多匹配”，除以 $\sqrt d$ 防止维度大时分数过大；再 softmax 把分数变成非负、和为 1 的权重；输出是 $V$ 的加权平均。
- **代入**：$Q=(1,0)$，$K_1=(1,1)$、$K_2=(0,1)$，$d=2$：$s_1=1/\sqrt2\approx0.707$，$s_2=0$；softmax 后 $w_1\approx0.669$、$w_2\approx0.331$。
- **边界**：不除以 $\sqrt d$，大维度点积会饱和，softmax 接近 one-hot；时间序列要先掩码 $j>t$ 为 $-\infty$ 再 softmax；先 softmax 再置零，权重和不再为 1。
- **用途/误读**：用于让模型按相关性分配注意力；权重不是因果重要性，也不能直接当因子暴露。

权重非负、和为 1；输出是值的加权平均。

### 7.3 因果掩码

时间序列里，时刻 $t$ 只能看 $j\le t$。做法：把 $j>t$ 的分数设为 $-\infty$，再 softmax。**先掩码，再归一化**；先归一化再置零，权重不会自动加回 1。

### 7.4 代码：注意力权重



In [7]:

import numpy as np

q = np.array([1.0, 0.0])
K = np.array([[np.sqrt(2), 0.0], [0.0, 1.0]])
V = np.array([2.0, -1.0])

scores = K @ q / np.sqrt(2)
weights = np.exp(scores - scores.max())
weights = weights / weights.sum()
print("权重", weights.round(3), "输出", round(float(weights @ V), 3))

logits = np.ones((2, 2))
mask = np.triu(np.ones((2, 2), dtype=bool), 1)
logits = np.where(mask, -np.inf, logits)
mass = np.exp(logits - logits.max(axis=1, keepdims=True))
causal = mass / mass.sum(axis=1, keepdims=True)
print("因果权重\n", causal.round(2))


权重 [0.731 0.269] 输出 1.193
因果权重
 [[1.  0. ]
 [0.5 0.5]]



**读数与应用**：第一条键更匹配查询，权重约 0.73；因果掩码后，第一行只能看自己，第二行可以看两条。真实 Transformer 的 Q/K/V 是学出来的，不是手定的。

**常见错误**：

1. 先 softmax 再掩码，剩余权重不归一。
2. 掩码只加在输出层，输入特征已经含未来信息。
3. 把注意力权重当因果效应。权重高不等于“导致”。

**停下来想一想**：为什么注意力机制在金融时序里容易泄漏？

<details><summary>对照你的思路</summary>

因为一条样本可能包含多个时间点，若不加因果掩码，时刻 $t$ 的输出会读到 $t+1$ 的键/值；此外，特征本身可能用了未来修订数据。掩码只能保护张量顺序，保护不了“数据在采集时已经含未来”。

</details>

---



## 8. 强化学习：行动改变未来状态

### 8.1 四个词

- 状态 $s$：当前能看到的信息（持仓、价格、波动）；
- 动作 $a$：改仓、平仓、持有；
- 奖励 $r$：当期损益/成本；
- 策略 $\pi(a|s)$：在状态 $s$ 下选动作的规则。

### 8.2 折扣与 Q 值

$$Q(s,a)=r+\gamma E[V(s')].$$

$\gamma$ 是折扣：越接近 1，越重视未来；越接近 0，越短视。

### 8.3 贝尔曼

$$V(s)=\max_a\left[r(s,a)+\gamma V(s')\right].$$

### 8.4 代码：折扣的作用



In [8]:

import numpy as np

reward = np.array([1.0, -0.2])       # 持有、平仓的即时奖励
next_value = np.array([-2.0, 0.0])   # 下一状态价值

for gamma in [0.0, 0.5, 0.9]:
    q = reward + gamma * next_value
    print("gamma", gamma, "Q", q, "选择", ["持有", "平仓"][q.argmax()])


gamma 0.0 Q [ 1.  -0.2] 选择 持有
gamma 0.5 Q [ 0.  -0.2] 选择 持有
gamma 0.9 Q [-0.8 -0.2] 选择 平仓



**读数与应用**：$\gamma=0$ 时只看眼前，选持有；$\gamma=0.9$ 时，持有的未来价值很差，选平仓。折扣改变了“短视/远视”。

**常见错误**：

1. 模拟器让策略用下一根收盘成交，却在当前就看见它。
2. 奖励设计只含收益，不含风险/成本。
3. 用历史回放当“环境”，但历史不能对动作做出反应。

**停下来想一想**：为什么强化学习在金融里很难做？

<details><summary>对照你的思路</summary>

因为真实市场是非平稳的、反事实不可观测：你做了 A，就看不到做 B 的结果；历史回放不能对动作做出反应。模拟器的假设（成交、冲击、对手行为）决定了策略学到什么；模拟器错了，策略就会利用漏洞。

</details>

---



## 9. 金融 ML 的特殊性与向前滚动验证

### 9.1 三个敌人

1. **低信噪比**：可预测部分很小，容易拟合噪声；
2. **非平稳**：关系会变，训练集和测试集分布不同；
3. **泄漏**：特征、标签、验证方式，任何一处用了未来信息。

### 9.2 标签重叠

若标签是“未来 5 天收益”，相邻两天的样本共享 4 天行情。随机 K 折会把“同一段行情”分到训练和测试，导致泄漏。

### 9.3 Purged / Embargo

做法：

- 按时间切分，不用随机打乱；
- 测试段前后各删掉“标签窗口”长度的样本（purge）；
- 再留一小段 embargo，防止特征窗口跨边界。

### 9.4 代码：Purged 切分



In [9]:

import numpy as np

n, horizon = 20, 3
for test_start in [0, 5, 10, 15]:
    test = np.arange(test_start, min(test_start + 5, n))
    purge = np.arange(max(0, test_start - horizon), test_start)
    train = np.setdiff1d(np.arange(n), np.r_[test, purge])
    print("测试", test[0], "到", test[-1], "训练数", len(train), "purge", purge)


测试 0 到 4 训练数 15 purge []
测试 5 到 9 训练数 12 purge [2 3 4]
测试 10 到 14 训练数 12 purge [7 8 9]
测试 15 到 19 训练数 12 purge [12 13 14]



**读数与应用**：测试段前面的 3 个样本被 purge 掉，因为它们标签窗口和测试段重叠。真实研究还要在测试段后面留 embargo，防止特征窗口跨边界。

**常见错误**：

1. 随机 K 折。时间序列必须按时间切。
2. 只 purge 前面，不 embargo 后面。
3. 用测试集反复调参；测试集变成“第二个训练集”。

**停下来想一想**：为什么“随机打乱”在时间序列里是灾难？

<details><summary>对照你的思路</summary>

随机打乱会把未来样本放进训练集，模型学到“未来信息”；测试集又包含过去的样本，评估虚高。时间序列的分布随时间变化，随机切分不能模拟“用过去预测未来”。必须用时间切分 + purge/embargo。

</details>

---



## 本章小结

### 一张公式表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 偏差-方差 | 误差=偏差²+方差+噪声 | 模型选择 |
| 线性回归 | $y=X\beta+\epsilon$ | 基准 |
| 激活 | ReLU $\max(0,z)$ | 非线性 |
| Gini | $1-\sum p_k^2$ | 树切分 |
| SVM | $\min\frac12||w||^2+C\sum\xi$ | 最大间隔 |
| KMeans | $\min\sum||x-\mu_k||^2$ | 聚类 |
| 注意力 | $\text{softmax}(QK^T/\sqrt d)V$ | 加权 |
| Q 值 | $r+\gamma E[V(s')]$ | 强化学习 |
| Purge | 删除标签窗口重叠样本 | 防泄漏 |

### 八句话

1. 训练误差低不等于样本外好。
2. 金融信噪比低，简单模型 + 正则往往更稳。
3. 没有激活函数，多层网络等价于一层。
4. 树越深越容易记住噪声。
5. SVM 的 $C$ 越大，正则越弱。
6. 聚类前先标准化。
7. 注意力要先掩码再 softmax。
8. 时间序列验证要按时间切，purge + embargo。

---

## 本章习题

**1.** 模型 A 训练 MSE 0.01、测试 MSE 0.10；模型 B 训练 0.05、测试 0.06。哪个更可能过拟合？

**2.** 偏差大、方差小，通常是什么问题？怎么处理？

**3.** 为什么 XOR 不能用一条直线分开？

**4.** 四个样本标签 0、0、1、1。原节点 Gini 是多少？按中间切开后 Gini 是多少？

**5.** SVM 的 $C$ 从 0.1 增到 100，间隔和过拟合风险会怎样变？

**6.** 两个资产价格 10 和 1000，成交量 100 和 200。直接聚类会有什么问题？

**7.** 注意力里，为什么先 softmax 再掩码是错的？

**8.** 标签是未来 5 天收益，训练/测试按时间切。为什么测试段前面要 purge 掉 5 天？

<details><summary>参考答案</summary>

**1.** A 更可能过拟合：训练误差极低、测试误差高，差距大。

**2.** 欠拟合。增加模型复杂度/特征，减少正则，检查特征是否足够。

**3.** XOR 的正类和负类不在一条直线的两侧；任何直线都会分错至少一个点。

**4.** 原节点两类各半，Gini $=1-(0.5^2+0.5^2)=0.5$；切开后每边纯，Gini $=0$。

**5.** $C$ 增大，间隔变窄、误分减少，过拟合风险上升；$C$ 减小，间隔变宽、偏差上升。

**6.** 价格和成交量的尺度差异大，距离被价格/成交量主导；需要标准化，并用收益/波动等可比特征。

**7.** 先 softmax 后掩码，被置零的权重不会重新归一，剩余权重和小于 1；应先掩码再 softmax。

**8.** 测试段前 5 天的标签窗口跨入测试段，和测试样本共享行情；不 purge 会把测试信息泄漏到训练。

</details>

---

## 参考资料

- Hastie, T., Tibshirani, R. & Friedman, J. *The Elements of Statistical Learning*（统计学习）。
- Murphy, K. P. *Machine Learning: A Probabilistic Perspective*（ML 全景）。
- López de Prado, M. *Advances in Financial Machine Learning*（purged CV、金融 ML 陷阱）。
- Vaswani, A. et al. *Attention Is All You Need*（注意力原论文）。
